In [ ]:
with first_push as (
    select
        contact_id,
        min(to_date(substr(campaign_name, 1, 8), 'YYYYMMDD')) as first_push_dt,
        max(is_sent::int) as is_sent,
        max(is_delivered::int) as is_delivered,
        max(is_opened::int) as is_opened
    from push_act
    group by contact_id
),

subscr_min as (
    select
        contact_id,
        min(subscr_date_act::date) as first_subscr_date
    from subscr_status
    group by contact_id
),

base as (
    select
        cc.client_id,
        cc.campaigns_cnt,
        coalesce(fp.is_sent, 0) as is_sent,
        coalesce(fp.is_delivered, 0) as is_delivered,
        coalesce(fp.is_opened, 0) as is_opened,

        case
            when coalesce(fp.is_opened, 0) = 1
             and (
                sm.first_subscr_date is null
                or sm.first_subscr_date < fp.first_push_dt
             )
            then 1 else 0
        end as opened_not_subscr

    from client_cohorts cc

    left join first_push fp
        on cc.client_id = fp.contact_id

    left join subscr_min sm
        on cc.client_id = sm.contact_id
),

agg as (
    select
        campaigns_cnt,
        count(*) as total_clients,
        sum(is_sent) as sent_clients,
        sum(is_delivered) as delivered_clients,
        sum(is_opened) as opened_clients,
        sum(opened_not_subscr) as opened_not_subscr_clients
    from base
    group by campaigns_cnt
)

select
    campaigns_cnt,
    total_clients,
    sent_clients,
    delivered_clients,
    opened_clients,
    opened_not_subscr_clients,

    round(sent_clients * 100.0 / nullif(total_clients, 0), 1) as sent_pct,
    round(delivered_clients * 100.0 / nullif(total_clients, 0), 1) as delivered_pct,
    round(opened_clients * 100.0 / nullif(total_clients, 0), 1) as opened_pct,
    round(opened_not_subscr_clients * 100.0 / nullif(opened_clients, 0), 1) as opened_not_subscr_pct_from_opened,
    round(opened_not_subscr_clients * 100.0 / nullif(total_clients, 0), 1) as opened_not_subscr_pct_total

from agg
order by campaigns_cnt;